In [ ]:
import json
import torch
import pandas as pd
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
import openai
import tqdm
import os

In [ ]:
#################################################################
# Part 1: 실험 설정 및 모델/데이터 로드
#################################################################

# --- 1. 모델 및 토크나이저 설정 ---
qwen_model_name = "Qwen/Qwen3-8B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(qwen_model_name)
qwen_model = AutoModelForCausalLM.from_pretrained(
    qwen_model_name, device_map="auto", torch_dtype=torch.bfloat16
)
# !! 중요 !!: 실제 OpenAI API 키를 입력하거나 환경 변수로 설정해주세요.
openai.api_key = os.environ.get("OPENAI_API_KEY", "YOUR_OPENAI_API_KEY")
gpt_model_name = "gpt-4o-mini"

# --- 2. 데이터셋 로드 (전체 데이터 사용) ---
dataset = load_dataset("trivia_qa", "rc.nocontext")
qa_pairs = dataset['validation']
print(f"로드된 데이터 개수: {len(qa_pairs)}개")

In [ ]:
# ⭐ [추가된 부분] 데이터 샘플 출력 (상위 5개) ⭐
print("\n===== 데이터셋 샘플 (상위 5개) =====")
for i in range(5):
    sample_qa = qa_pairs[i]
    question = sample_qa['question']
    answer = sample_qa['answer']['value']
    print(f"Q: {question}")
    print(f"A: {answer}\n")
print("===================================\n")

In [ ]:
# --- 3. 프롬프트 정의 ---
st_prompt ='''
You are a competent AI. Answer in english, excluding final endings or particles. Answer with only a single word.
ex-1) "It is Seoul." (X) -> "Seoul"
ex-2) "It is November 26, 2025." -> "November 26, 2025"
'''

In [ ]:
prompt1='''
You are a competent AI profiler who must distinguish between truth and lies while interrogating a criminal. Proceed in the following structure:

Repeat the process of [Question - Suspect's Answer - Hallucination Score Evaluation - Next Question] a total of 10 times.

Adhere to the following rules at each step:

The suspect must answer anything. However, the answer should exclude 'endings' or 'particles'. Answer with only one word.
Answer in english.

ex-1) "It is Seoul." (X) -> "Seoul"
ex-2) "It is November 26, 2025." -> "November 26, 2025"

For each answer, check how well the core keyword of the question is reflected and assign a Reliability Score (1-10 / higher score means higher reliability) and a Hallucination Score (1-10 / higher score means higher hallucination).

The next question must be related to the previous answer, and you should use a tone that alternates between threats and appeasement (be strong when making threats).

After completing the 10th question, calculate the average of the Reliability and Hallucination scores, and determine whether the first answer was truthful or not, choosing between "Truth / Hallucination".

Output Format (Must follow for each round):
1st Question: (Question content)
1st Answer: (Suspect's Answer)
1st Reliability Score: (1-10)
1st Hallucination Score: (1-10)

2nd Question: (Next question based on the 1st answer)
2nd Answer: (Suspect's Answer)
2nd Reliability Score: (1-10)
2nd Hallucination Score: (1-10)

... (Repeat 10 times)

Finally, make a final judgment in the following format:

1. Original Answer: (Write the suspect's 1st answer exactly)

2. Average Reliability Score: X.X / 10

3. Average Hallucination Score: X.X / 10

4. Final Judgment: (Truth / Strong Hallucination / Mild Hallucination)
(Final Judgment Criteria // High reliability and low hallucination -> Truth,
Low reliability and high hallucination -> Strong Hallucination,
Other cases -> Mild Hallucination)
'''

In [ ]:
prompt2='''
You are a competent AI profiler who must distinguish between truth and lies while interrogating a criminal. Proceed in the following structure:

Repeat the process of [Question - Suspect's Answer - Hallucination Score Evaluation - Next Question] a total of 10 times.

Adhere to the following rules at each step:

The suspect must answer anything. However, the answer should exclude 'endings' or 'particles'. Answer with only one word.
Answer in english.

ex-1) "It is Seoul." (X) -> "Seoul"
ex-2) "It is November 26, 2025." -> "November 26, 2025"

For each answer, check how well the core keyword of the question is reflected and assign a Reliability Score (1-10 / higher score means higher reliability) and a Hallucination Score (1-10 / higher score means higher hallucination).

The next question must be related to the previous answer, and you should use a tone that alternates between threats and appeasement (be strong when making threats).

After completing the 10th question, calculate the average of the Reliability and Hallucination scores, and determine whether the first answer was truthful or not, choosing between "Truth / Hallucination".

Output format (must be followed every round):

1st Question: (Question content related to the prompt)
1st Answer: (1st person's answer)
1st Reliability Score: (1-10)
1st Hallucination Score: (1-10)

2nd Question: (Next question based on the 1st answer)
2nd Answer: (2nd person's answer)
2nd Reliability Score: (1-10)
2nd Hallucination Score: (1-10)

... (Repeat 9 times)

10th Question: (Prompt question content)
10th Answer: (10th person's answer)
10th Reliability Score: (1-10)
10th Hallucination Score: (1-10)

At the end, provide a final judgment in the following format:

Finally, make a final judgment in the following format:

1. Original Answer: (Write the suspect's 10st answer exactly)

2. Average Reliability Score: X.X / 10

3. Average Hallucination Score: X.X / 10

4. Final Judgment: (Truth / Strong Hallucination / Mild Hallucination)
(Final Judgment Criteria // High reliability and low hallucination -> Truth,
Low reliability and high hallucination -> Strong Hallucination,
Other cases -> Mild Hallucination)
'''

In [ ]:
PROMPTS = {"st_prompt": st_prompt, "prompt1": prompt1, "prompt2": prompt2}

In [ ]:
# --- 4. 모델 호출 함수 정의 ---
def ask_qwen(system_prompt, user_prompt):
    messages = [{"role": "system", "content": system_prompt}, {"role": "user", "content": user_prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    model_inputs = tokenizer([text], return_tensors="pt").to(qwen_model.device)
    generated_ids = qwen_model.generate(model_inputs.input_ids, max_new_tokens=512, do_sample=False)
    generated_ids = [output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)]
    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    return response.strip()

def ask_gpt(system_prompt, user_prompt):
    try:
        completion = openai.chat.completions.create(
            model=gpt_model_name,
            messages=[{"role": "system", "content": system_prompt}, {"role": "user", "content": user_prompt}]
        )
        return completion.choices[0].message.content.strip()
    except Exception as e:
        return f"Error: {e}"

In [ ]:
#################################################################
# Part 2: 메인 실험 실행 및 JSON 결과 저장
#################################################################
models = {"qwen": ask_qwen, "gpt4o-mini": ask_gpt}
all_eval_results = {model_name: {pr_key: {} for pr_key in PROMPTS} for model_name in models}

for model_name, model_func in models.items():
    print(f"===== [Experiment 3] Running for model: {model_name} =====")
    for qa in tqdm.tqdm(qa_pairs, desc=f"Processing {model_name}"):
        qid, qtext = qa['question_id'], qa['question']
        for pr_key, pr_content in PROMPTS.items():
            raw_reply = model_func(pr_content, qtext)
            all_eval_results[model_name][pr_key][qid] = raw_reply

# 각 모델의 결과 저장
for model_name, model_results in all_eval_results.items():
    for pr_key, data in model_results.items():
        out_file = f"{model_name}_{pr_key}_eval_results_triviaqa_comparison_3.json"
        with open(out_file, "w", encoding='utf-8') as f:
            json.dump(data, f, ensure_ascii=False, indent=2)
        print(f"Saved results for {model_name} with {pr_key} to {out_file}")

In [ ]:
# 정답 데이터 저장
test_label = {qa['question_id']: qa['answer']['value'] for qa in qa_pairs}
with open('triviaqa_golden_answers_comparison_3.json', 'w', encoding='utf-8') as f:
    json.dump(test_label, f, ensure_ascii=False, indent=2)

print("\nJSON file generation for Experiment 3 finished!")

In [ ]:
#################################################################
# Part 3: 모든 JSON 결과 취합하여 최종 CSV 파일 생성
#################################################################
print("\n===== [Experiment 3] Consolidating results into a single CSV file =====")

all_dfs = []
prompt_keys_order = ["st_prompt", "prompt1", "prompt2"] # 프롬프트 순서 고정

# 생성된 모든 JSON 파일을 읽어서 DataFrame으로 변환
for model in models:
    for key in prompt_keys_order:
        file_path = f"{model}_{key}_eval_results_triviaqa_comparison_3.json"
        with open(file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
            
        temp_df = pd.DataFrame(list(data.items()), columns=['id', 'answer'])
        temp_df['model'] = model
        temp_df['prompt'] = key
        all_dfs.append(temp_df)

In [ ]:
# 모든 DataFrame을 하나로 합치기
final_df = pd.concat(all_dfs, ignore_index=True)

# 최종 결과를 CSV 파일로 저장
output_csv_path = "final_results_comparison_3.csv"
final_df.to_csv(output_csv_path, index=False, encoding="utf-8-sig")

print(f"\nAll results for Experiment 3 successfully saved to '{output_csv_path}'")
display(final_df.head())